import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from sklearn.impute import SimpleImputer

In [ ]:
Note that we dont have noise in k means inherently

n_init‘auto’ or int, default=”auto”
Number of time the k-means algorithm will be run with different centroid seeds. The final results will be the best output of n_init consecutive runs in terms of inertia.

When n_init='auto', the number of runs depends on the value of init: 10 if using init='random' or init is a callable; 1 if using init='k-means++' or init is an array-like.

In [ ]:

# Pivot data to get 24-hour profile for each device_globalid
pivot_data = df.pivot_table(index='device_globalid', columns='hour_of_day', values='avg_kw_per_hour', aggfunc='mean').fillna(0)

# Preprocess data- replace the nulls with mean
features = pivot_data.columns
imputer = SimpleImputer(strategy='mean')
pivot_data[features] = imputer.fit_transform(pivot_data[features])

# Scale the data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(pivot_data[features])

# Perform K-means with 6 clusters
optimal_k = 6
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=5)
clusters = kmeans.fit_predict(scaled_data)

# Add cluster labels to the pivot data
pivot_data['Cluster'] = clusters

# Visualize clusters using PCA
pca = PCA(n_components=2)
reduced_data = pca.fit_transform(scaled_data)
plt.figure(figsize=(8, 5))
scatter = plt.scatter(reduced_data[:, 0], reduced_data[:, 1], c=pivot_data['Cluster'], cmap='viridis', alpha=0.6)
plt.title(f'K-means Clustering with {optimal_k} Clusters (PCA-reduced)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.colorbar(scatter, label='Cluster')
plt.grid(True)
plt.show()

# Print cluster sizes
print(f"\nCluster sizes for k={optimal_k}:")
print(pivot_data['Cluster'].value_counts())

# Analyze cluster characteristics
print("\nCluster centroids (original scale):")
centroids = scaler.inverse_transform(kmeans.cluster_centers_)
centroid_df = pd.DataFrame(centroids, columns=features)
print(centroid_df)

cluster_centers_ndarray of shape (n_clusters, n_features)
Coordinates of cluster centers. If the algorithm stops before fully converging (see tol and max_iter), these will not be consistent with labels_.

labels_ndarray of shape (n_samples,)
Labels of each point

inertia_float
Sum of squared distances of samples to their closest cluster center, weighted by the sample weights if provided.

n_iter_int
Number of iterations run.

n_features_in_int
Number of features seen during fit.